### Allscripts Sunrise (SCM) - Care Site Hydration

**Source Tables:**
- _exponent._bronze_allscripts_scm_prod_01.dbo_cv3location

**Strategy:**
- Extract location/facility data from dbo_cv3location
- Map TypeCode to place of service concepts
- Use GUID as care_site_source_value
- Populate care_site_name from Name field
- Filter to Active locations only

**Note:**
Care site is a reference table - loaded once, rarely changes

In [0]:
source = 'allscripts_scm'

# Transformation

In [0]:
silver_care_site_df = spark.sql(f'''
SELECT 
  loc.Name AS care_site_name,
  COALESCE(pos_concept.omop_concept_id, 0) AS place_of_service_concept_id,
  NULL AS location_id,
  CONCAT('{source}', ' | ', loc.GUID) AS care_site_source_value,
  loc.Code AS place_of_service_source_value,
  '{source}' AS source_system
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3location loc
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept pos_concept
  ON pos_concept.source_id = CAST(loc.PlaceOfServiceID AS STRING)
  AND pos_concept.domain_id = 'Place of Service'
  AND pos_concept.source_system = '{source}'
WHERE loc.Active = TRUE
  AND loc.Name IS NOT NULL
''')

display(silver_care_site_df)
silver_care_site_df.createOrReplaceTempView("silver_care_site")

# Merge to Silver

In [0]:
%sql
MERGE INTO _exponent.omop_silver.care_site AS t
USING (
  SELECT * FROM silver_care_site
) AS s
ON t.care_site_source_value = s.care_site_source_value

WHEN MATCHED THEN UPDATE SET
  t.care_site_name = s.care_site_name,
  t.place_of_service_concept_id = s.place_of_service_concept_id,
  t.location_id = s.location_id,
  t.place_of_service_source_value = s.place_of_service_source_value

WHEN NOT MATCHED THEN INSERT (
  care_site_name,
  place_of_service_concept_id,
  location_id,
  care_site_source_value,
  place_of_service_source_value,
  source_system
)
VALUES (
  s.care_site_name,
  s.place_of_service_concept_id,
  s.location_id,
  s.care_site_source_value,
  s.place_of_service_source_value,
  s.source_system
);

# Populate Mapping Table

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_care_site (
  care_site_source_value,
  active_flag
)
SELECT 
  care_site_source_value,
  TRUE
FROM _exponent.omop_silver.care_site
WHERE care_site_source_value NOT IN (
  SELECT care_site_source_value 
  FROM _exponent.omop_mapping.source_to_care_site
  WHERE active_flag = TRUE
);

# Merge to Gold

In [0]:
%sql
MERGE INTO _exponent.omop.care_site AS gold
USING (
  SELECT 
    source_to_care_site.care_site_id,
    s.care_site_name,
    s.place_of_service_concept_id,
    s.location_id,
    s.care_site_source_value,
    s.place_of_service_source_value
  FROM _exponent.omop_silver.care_site s
  JOIN _exponent.omop_mapping.source_to_care_site
    ON source_to_care_site.care_site_source_value = s.care_site_source_value
    AND source_to_care_site.active_flag = TRUE
) AS src
ON gold.care_site_id = src.care_site_id

WHEN MATCHED THEN UPDATE SET
  gold.care_site_name = src.care_site_name,
  gold.place_of_service_concept_id = src.place_of_service_concept_id,
  gold.location_id = src.location_id,
  gold.care_site_source_value = src.care_site_source_value,
  gold.place_of_service_source_value = src.place_of_service_source_value

WHEN NOT MATCHED THEN INSERT (
  care_site_id,
  care_site_name,
  place_of_service_concept_id,
  location_id,
  care_site_source_value,
  place_of_service_source_value
)
VALUES (
  src.care_site_id,
  src.care_site_name,
  src.place_of_service_concept_id,
  src.location_id,
  src.care_site_source_value,
  src.place_of_service_source_value
);